# 02: Train and evaluate the BC steering network

Loads the k-fold dataset built at the end of `data-cleaning.ipynb` (`processed/bc_v1_folds.npz`), trains the
steering MLP once per fold, compares it with simple baselines, and finally trains one model on all the data.

This notebook only loads the small processed file (~1 MB), not the raw runs. **Shut down the data-cleaning
kernel before running it**: this machine has little RAM, and torch itself takes a lot.

The model and training loop live in `src/rl_driving/bc.py` (the PPO and car code will need the same network).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
import torch

from rl_driving.preprocess import PREPROC_VERSION, STEER_MAX
from rl_driving.bc import SteeringMLP, train_bc, predict, save_model

torch.set_num_threads(2)          # this machine has 2 cores; don't starve VS Code

ds = np.load(f'../processed/bc_v{PREPROC_VERSION}_folds.npz')
X, y, drift = ds['X'], ds['y'], ds['drift']
train_masks, val_masks = ds['train_masks'], ds['val_masks']
N_FOLDS = len(train_masks)
print(f'{len(y)} samples, X {X.shape}, runs {list(ds["runs"])}, {N_FOLDS} folds, '
      f'chunks of {float(ds["chunk_len"])} s, gap {float(ds["gap"])} s')

## Metrics
The model predicts **normalised** steering (rad / 0.4189). For reading results, errors are converted back to **radians**:
- **RMSE** (rad): typical error, punishes big misses. The wall follower's steering spans about ±0.3 rad.
- **MAE** (rad): average absolute error.
- **R²**: 1 = perfect, 0 = no better than always predicting the mean steering.
- **MAE on drift rows**: errors during drift, where the expert is correcting. These rows hold the recovery behaviour,
  so the model must do well on them, not just on normal driving.

In [ ]:
def metrics(pred, true, drift_rows):
    err = (pred - true) * STEER_MAX                   # back to radians
    ss_res = np.sum((pred - true) ** 2)
    ss_tot = np.sum((true - true.mean()) ** 2)
    return {
        'rmse': float(np.sqrt(np.mean(err ** 2))),
        'mae': float(np.mean(np.abs(err))),
        'r2': float(1 - ss_res / ss_tot),
        'mae_drift': float(np.mean(np.abs(err[drift_rows == 1]))) if drift_rows.any() else np.nan,
    }

## Baselines
A score means little on its own, so the MLP is compared with:
- **zero**: always steer straight
- **mean**: always predict the training mean steering
- **linear**: ridge regression on the same 218 inputs. If the MLP isn't clearly better than this, the extra complexity isn't paying off.

In [ ]:
def fit_ridge(X_tr, y_tr, lam=1e-2):
    A = np.c_[X_tr, np.ones(len(X_tr))]
    w = np.linalg.solve(A.T @ A + lam * np.eye(A.shape[1]), A.T @ y_tr)
    return lambda X_: np.c_[X_, np.ones(len(X_))] @ w

## K-fold training
For each fold: train on the train mask, early-stop on the val mask (keep the weights from the best val epoch),
and score the MLP and the baselines on that fold's val rows.

Settings: 2×256 tanh MLP, Adam, learning rate 1e-3 with cosine decay down to 1e-5, weight decay 1e-4, batch 256,
up to 300 epochs, stop after 60 epochs without val improvement.
A first try with a fixed learning rate and 200 epochs was still improving at the end and its loss curves were spiky.
The decay lets the weights settle. Each fold takes ~2–3 min on this 2-core machine (the whole notebook ~15 min).

In [ ]:
EPOCHS, LR, WEIGHT_DECAY, BATCH, PATIENCE = 300, 1e-3, 1e-4, 256, 60

results, histories, val_preds = [], [], {}
for k in range(N_FOLDS):
    tr, va = train_masks[k], val_masks[k]
    model, hist = train_bc(X[tr], y[tr], X[va], y[va], epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
                           batch_size=BATCH, patience=PATIENCE, seed=k)
    pred = predict(model, X[va])
    val_preds[k] = pred
    histories.append(hist)

    ridge = fit_ridge(X[tr], y[tr])
    for name, p in [('mlp', pred), ('linear', ridge(X[va])),
                    ('mean', np.full(va.sum(), y[tr].mean())), ('zero', np.zeros(va.sum()))]:
        results.append({'fold': k, 'model': name, **metrics(p, y[va], drift[va])})
    print(f'fold {k}: best epoch {hist["best_epoch"]:3d}, val RMSE {results[-4]["rmse"]:.4f} rad, R² {results[-4]["r2"]:.3f}')

## Results across folds
Mean ± standard deviation over the 5 folds. The spread shows how much the score depends on which chunks were held out.

In [ ]:
print(f'{"model":8s} {"RMSE [rad]":>16s} {"MAE [rad]":>16s} {"R²":>14s} {"MAE drift [rad]":>17s}')
for name in ['mlp', 'linear', 'mean', 'zero']:
    rows = [r for r in results if r['model'] == name]
    cols = []
    for key in ['rmse', 'mae', 'r2', 'mae_drift']:
        v = np.array([r[key] for r in rows])
        cols.append(f'{np.nanmean(v):.4f} ± {np.nanstd(v):.4f}')
    print(f'{name:8s} {cols[0]:>16s} {cols[1]:>16s} {cols[2]:>14s} {cols[3]:>17s}')

print('\nper fold (MLP):')
for r in results:
    if r['model'] == 'mlp':
        print(f'  fold {r["fold"]}: RMSE {r["rmse"]:.4f}  MAE {r["mae"]:.4f}  R² {r["r2"]:.3f}  MAE drift {r["mae_drift"]:.4f}')

## Reading the results: the noise floor
The linear baseline does about as well as the MLP, or slightly better. That's expected for **this** expert:
- wall_follow is a PID controller on a few lidar distances, so its steering is close to a **linear** function of the scan.
  A linear model is the right shape for it.
- Its labels are **noisy**: consecutive labels (26 ms apart) differ by ~0.034 rad (std), because the PID reacts to lidar noise.
  No model can predict that jitter from the inputs. Even ridge regression only fits its own *training* rows to ~0.022 rad.
  That is roughly the **noise floor**, and both models are near it.
- The gap between them is a few thousandths of a rad (a fraction of a degree), small compared with the ±0.3 rad steering range.

So on wall_follow data, the MLP doesn't need to beat linear. The pipeline works and the MLP learns the expert, including
during drift. The comparison matters again with the **cv-nav** expert, which is not a simple function of the lidar:
there the MLP should pull ahead. If it doesn't, that's a sign to look at the inputs.

In [ ]:
S = STEER_MAX
for k in range(N_FOLDS):
    tr = train_masks[k]
    ridge = fit_ridge(X[tr], y[tr])
    print(f'fold {k}: linear fit on its own train rows: RMSE {np.sqrt(np.mean((ridge(X[tr]) - y[tr]) ** 2)) * S:.4f} rad')
print(f'label jitter between consecutive scans: std {np.std(np.diff(y)) * S:.4f} rad')

## Learning curves
Train and val MSE per epoch, one line per fold (log scale). The dot marks the best val epoch.
- Train keeps falling while val flattens or rises: the network starts memorising. Early stopping keeps the best point.
- A large, growing gap between train and val means overfitting: more data, more weight decay or a smaller net would help.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for k, h in enumerate(histories):
    color = plt.cm.tab10(k)
    ax.plot(h['train'], color=color, lw=1, ls='--', alpha=0.7)
    ax.plot(h['val'], color=color, lw=1.6, label=f'fold {k}')
    ax.plot(h['best_epoch'], h['val'][h['best_epoch']], 'o', color=color)
ax.set_yscale('log'); ax.set_xlabel('epoch'); ax.set_ylabel('MSE (normalised steering)')
ax.set_title('dashed = train, solid = val'); ax.legend(frameon=False, ncol=5, fontsize=8); ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

## Predictions vs expert on held-out data
For one fold: the expert's steering (label) and the MLP's prediction over time, on that fold's val rows only.
Drift periods are shaded. Look for whether the MLP follows the corners and the expert's corrections during drift,
and whether it lags or smooths the expert.

In [ ]:
k = 0
va = val_masks[k]
t_va, run_va = ds['t'][va], ds['run'][va]
runs_in_fold = np.unique(run_va)

fig, axes = plt.subplots(len(runs_in_fold), 1, figsize=(13, 3 * len(runs_in_fold)), squeeze=False)
for ax, r in zip(axes[:, 0], runs_in_fold):
    m = run_va == r
    tt = t_va[m]
    d = drift[va][m] == 1
    ax.fill_between(tt, -0.35, 0.35, where=d, color='#888888', alpha=0.2, lw=0, label='drift active')
    ax.plot(tt, y[va][m] * STEER_MAX, '.', ms=2.5, color='#2a78d6', label='expert (label)')
    ax.plot(tt, val_preds[k][m] * STEER_MAX, '.', ms=2.5, color='#eb6834', label='MLP prediction')
    ax.set_ylabel('steering [rad]'); ax.set_title(f'fold {k}, run {ds["runs"][r]} (val rows only)', fontsize=10)
    ax.grid(alpha=0.25); ax.legend(frameon=False, fontsize=8, ncol=3)
axes[-1, 0].set_xlabel('time [s]')
plt.tight_layout(); plt.show()

## Final model on all the data
K-fold was for **measuring**; the model we keep is trained on **all** rows. There's no val set left for early
stopping, so it trains for the median best epoch across the folds (+1, since epochs count from 0).
Saved to `checkpoints/bc_v1.pt` (gitignored) with the preprocessing version and the k-fold scores.
`rl_driving.bc.load_model` refuses to load it if the preprocessing version changes.

In [ ]:
final_epochs = int(np.median([h['best_epoch'] for h in histories])) + 1
final_model, _ = train_bc(X, y, epochs=final_epochs, lr=LR, weight_decay=WEIGHT_DECAY, batch_size=BATCH, seed=0)

mlp_rows = [r for r in results if r['model'] == 'mlp']
os.makedirs('../checkpoints', exist_ok=True)
ckpt_path = f'../checkpoints/bc_v{PREPROC_VERSION}.pt'
save_model(final_model, ckpt_path, epochs=final_epochs, runs=list(ds['runs']),
           kfold_rmse_rad=float(np.mean([r['rmse'] for r in mlp_rows])),
           kfold_r2=float(np.mean([r['r2'] for r in mlp_rows])))
print(f'trained on {len(y)} samples for {final_epochs} epochs, saved {ckpt_path}')
print(f'fit on its own training data: {metrics(predict(final_model, X), y, drift)}')

## What these numbers do and don't say
- They measure how well the network **copies the expert** on unseen stretches of the same track, one scan at a time.
- They **don't** show whether it can drive: in closed loop, small errors move the car into situations the expert never
  recorded, and errors can add up. That needs the sim (step 1 and 4) and then the car, with the safety node on.
- All of this is wall_follow data. The pipeline is ready for the cv-nav recordings: add them to `RUNS` in
  `data-cleaning.ipynb`, rebuild, and rerun this notebook.